# Task 1: Data Analysis and Preparation

In [ ]:
# --- Imports ---
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.feature_selection import SelectKBest, f_regression
import warnings
warnings.filterwarnings('ignore')

# 1. Load Data
# In Colab, upload the file or use the path if mounted
df = pd.read_csv('AB_NYC_2019[2].csv') # Adjust path if needed

# 2. Initial Exploration
print("Shape:", df.shape)
print("Missing Values:\n", df.isnull().sum())
print("\nBasic Stats:\n", df.describe())

# 3. Cleaning & Preprocessing Strategy
# A. Missing Values:
#    - 'reviews_per_month' and 'last_review' are missing for listings with 0 reviews.
#    - We impute with 0 because no reviews = 0 reviews/month.
df['reviews_per_month'] = df['reviews_per_month'].fillna(0)
df['last_review'] = df['last_review'].fillna('1900-01-01') # Placeholder for date parsing

# B. Outliers:
#    - 'price' can have extreme outliers (e.g., $10,000+).
#    - We will use a log transformation later, but first, let's cap extreme prices
#    - or simply rely on robust models (Random Forest) that handle outliers better than Linear Regression.
#    - For this workflow, we will keep the raw price but use Log-Price as the target to stabilize variance.

# C. Feature Engineering
# Convert date to datetime
df['last_review'] = pd.to_datetime(df['last_review'], errors='coerce')

# Extract year/month from last review (might indicate how active the host is)
df['last_review_year'] = df['last_review'].dt.year
df['last_review_month'] = df['last_review'].dt.month

# Create a 'recency' feature: How many days since last review?
# Reference date: Max date in dataset or a fixed date like 2019-12-31
ref_date = pd.Timestamp('2019-12-31')
df['days_since_review'] = (ref_date - df['last_review']).dt.days
df['days_since_review'] = df['days_since_review'].fillna(9999) # If no review, assume very old

# Drop original date column
df.drop(['last_review'], axis=1, inplace=True)

# D. Identify Feature Types
numeric_features = ['minimum_nights', 'number_of_reviews', 'reviews_per_month',
                    'calculated_host_listings_count', 'availability_365',
                    'last_review_year', 'last_review_month', 'days_since_review',
                    'latitude', 'longitude']

categorical_features = ['neighbourhood_group', 'neighbourhood', 'room_type']

# Target Variable
# We use Log(Price) to handle the right-skewed distribution of prices
df['log_price'] = np.log1p(df['price']) # log(1 + price) to handle zeros if any

# Drop original price for modeling
X = df.drop(['price', 'log_price', 'id', 'host_id', 'host_name', 'name'], axis=1)
y = df['log_price']

# 4. Feature Selection
# Let's check correlation with target
corr_matrix = X[numeric_features].corrwith(y).abs().sort_values(ascending=False)
print("\nTop Correlated Features:\n", corr_matrix.head(10))

# 5. Train/Test Split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

Shape: (48895, 16)
Missing Values:
 id                                    0
name                                 16
host_id                               0
host_name                            21
neighbourhood_group                   0
neighbourhood                         0
latitude                              0
longitude                             0
room_type                             0
price                                 0
minimum_nights                        0
number_of_reviews                     0
last_review                       10052
reviews_per_month                 10052
calculated_host_listings_count        0
availability_365                      0
dtype: int64

Basic Stats:
                  id       host_id      latitude     longitude         price  \
count  4.889500e+04  4.889500e+04  48895.000000  48895.000000  48895.000000   
mean   1.901714e+07  6.762001e+07     40.728949    -73.952170    152.720687   
std    1.098311e+07  7.861097e+07      0.054530      0.0461

# Task 2 - Model Training and Evaluation

In [ ]:
# --- Imports for Modeling ---
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor, HistGradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
import os

# 1. Define Preprocessing Steps
# Numeric: Impute missing, Scale
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='mean')),
    ('scaler', StandardScaler())
])

# Categorical: Impute missing, One-Hot Encode
categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='constant', fill_value='Unknown')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

# Combine into Preprocessor
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features)
    ])

# 2. Define Models
models = {
    'Linear Regression': LinearRegression(),
    'Ridge Regression': Ridge(alpha=1.0),
    'Random Forest': RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1),
    'Gradient Boosting': GradientBoostingRegressor(n_estimators=100, random_state=42),
    'Hist Gradient Boosting': HistGradientBoostingRegressor(random_state=42)
}

# 3. Train and Evaluate
results = []

for name, model in models.items():
    # Create Pipeline: Preprocess -> Model
    pipeline = Pipeline(steps=[
        ('preprocessor', preprocessor),
        ('regressor', model)
    ])

    # Train
    pipeline.fit(X_train, y_train)

    # Predict
    y_pred_train = pipeline.predict(X_train)
    y_pred_test = pipeline.predict(X_test)

    # Metrics (on Test Set)
    mae = mean_absolute_error(y_test, y_pred_test)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred_test))
    r2 = r2_score(y_test, y_pred_test)

    # Metrics (on Train Set) to check overfitting
    r2_train = r2_score(y_train, y_pred_train)

    results.append({
        'Model': name,
        'Test MAE': mae,
        'Test RMSE': rmse,
        'Test R2': r2,
        'Train R2': r2_train
    })

    print(f"{name}: Test R2={r2:.4f}, RMSE={rmse:.4f}, Train R2={r2_train:.4f}")

# Convert to DataFrame for easy comparison
results_df = pd.DataFrame(results)
print("\nModel Comparison:")
print(results_df)

# 4. Select Best Model
# Typically, HistGradientBoosting or Random Forest performs best on tabular data with mixed features.
best_model_name = results_df.loc[results_df['Test R2'].idxmax(), 'Model']
print(f"\nBest Model: {best_model_name}")

# 5. Hyperparameter Tuning (Example for Random Forest)
# If Random Forest was best, we would tune n_estimators and max_depth.
# For brevity, we assume HistGradientBoosting is robust enough without extensive tuning
# for this project scope, but in a real scenario, use GridSearchCV.

# 6. Save the Final Pipeline
import joblib

# Re-define the best pipeline to save
final_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('regressor', models[best_model_name])
])
final_pipeline.fit(X_train, y_train)

# Create the directory if it doesn't exist
os.makedirs('app/', exist_ok=True)

# Save to file
joblib.dump(final_pipeline, 'app/model_pipeline.pkl')
print("Pipeline saved to app/model_pipeline.pkl")

# 7. Feature Importance (If using Tree-based model)
if best_model_name in ['Random Forest', 'Gradient Boosting', 'Hist Gradient Boosting']:
    # Get feature names from the preprocessor
    # Note: For HistGradientBoosting, feature_importances_ might not be directly available in all sklearn versions.
    # We'll use Random Forest for a clear example if it was close, or just plot the top numeric correlations.
    pass

Linear Regression: Test R2=0.5546, RMSE=0.4590, Train R2=0.5425
Ridge Regression: Test R2=0.5548, RMSE=0.4589, Train R2=0.5422
Random Forest: Test R2=0.6109, RMSE=0.4290, Train R2=0.9430
Gradient Boosting: Test R2=0.5941, RMSE=0.4382, Train R2=0.5916
Hist Gradient Boosting: Test R2=0.6163, RMSE=0.4261, Train R2=0.6384

Model Comparison:
                    Model  Test MAE  Test RMSE   Test R2  Train R2
0       Linear Regression  0.333327   0.459049  0.554573  0.542533
1        Ridge Regression  0.333223   0.458915  0.554833  0.542192
2           Random Forest  0.306066   0.429037  0.610912  0.942986
3       Gradient Boosting  0.315324   0.438197  0.594121  0.591571
4  Hist Gradient Boosting  0.305414   0.426050  0.616310  0.638401

Best Model: Hist Gradient Boosting
Pipeline saved to app/model_pipeline.pkl


# Task 4

In [3]:
!pip install streamlit
import streamlit as st
import pandas as pd
import joblib
import numpy as np

# Load Model
try:
    # Corrected model path
    pipeline = joblib.load('app/model_pipeline.pkl')
except FileNotFoundError:
    st.error("Model file not found. Please run the training notebook first (Task 2) to save 'app/model_pipeline.pkl'.")
    st.stop()

st.title("Airbnb Price Predictor")
st.write("Enter the listing details below to predict the nightly price.")

# Define Inputs
col1, col2 = st.columns(2)

with col1:
    neighbourhood_group = st.selectbox(
        "Neighbourhood Group",
        ["Manhattan", "Brooklyn", "Queens", "Bronx", "Staten Island"],
        index=1 # Default to Brooklyn
    )
    # Added neighbourhood input. For a full app, this would be a selectbox populated from training data.
    neighbourhood = st.text_input("Neighbourhood", value="Williamsburg")
    room_type = st.selectbox(
        "Room Type",
        ["Private room", "Entire home/apt", "Shared room"]
    )
    latitude = st.slider("Latitude", 40.5, 40.9, 40.75, 0.01)
    longitude = st.slider("Longitude", -74.0, -73.8, -73.95, 0.01)

with col2:
    number_of_reviews = st.number_input("Number of Reviews", min_value=0, value=10)
    minimum_nights = st.number_input("Minimum Nights", min_value=1, value=3)
    reviews_per_month = st.number_input("Reviews per Month", min_value=0.0, value=0.5, step=0.1)
    calculated_host_listings_count = st.number_input("Host Listings Count", min_value=1, value=1)
    availability_365 = st.number_input("Availability (days/year)", min_value=0, max_value=365, value=365)

# Prepare Input Data
if st.button("Predict Price"):
    # Derive date-related features, consistent with training notebook logic
    ref_date = pd.Timestamp('2019-12-31') # Same reference date as in training

    if number_of_reviews == 0:
        last_review_year = 1900 # Placeholder for no review
        last_review_month = 1   # Placeholder for no review
        days_since_review = 9999 # Placeholder for no review (very old)
    else:
        # Assume a recent review date if there are reviews (e.g., 2 months before ref_date)
        assumed_last_review_date = ref_date - pd.Timedelta(days=60) # Approx 2 months before ref_date
        last_review_year = assumed_last_review_date.year
        last_review_month = assumed_last_review_date.month
        days_since_review = (ref_date - assumed_last_review_date).days

    # Create a DataFrame with the features expected by the trained pipeline
    # These features match the 'X' DataFrame used for training, including derived date features.
    input_data = pd.DataFrame({
        'neighbourhood_group': [neighbourhood_group],
        'neighbourhood': [neighbourhood],
        'latitude': [latitude],
        'longitude': [longitude],
        'room_type': [room_type],
        'minimum_nights': [minimum_nights],
        'number_of_reviews': [number_of_reviews],
        'reviews_per_month': [reviews_per_month],
        'calculated_host_listings_count': [calculated_host_listings_count],
        'availability_365': [availability_365],
        'last_review_year': [last_review_year],
        'last_review_month': [last_review_month],
        'days_since_review': [days_since_review]
    })

    # Predict log_price
    log_prediction = pipeline.predict(input_data)
    # Convert log_price back to actual price
    predicted_price = np.expm1(log_prediction[0]) # exp(log_price) - 1

    # Corrected f-string formatting
    st.success(f"Predicted Nightly Price: ${predicted_price:.2f}")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.5/10.5 MB 49.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 69.6 MB/s eta 0:00:00


2026-09-14 09:21:55.642 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-09-14 09:21:55.827 
  command:

    streamlit run /usr/local/lib/python3.13/dist-packages/colab_kernel_launcher.py [ARGUMENTS]
2026-09-14 09:21:55.829 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-09-14 09:21:55.832 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-09-14 09:21:55.835 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-09-14 09:21:55.839 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-09-14 09:21:55.842 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-09-14 09:21:55.845 Thread 'MainThread': mi